# 02 — The "random" in the forest: decorrelating the trees

*Chapter 06 — Random Forests · Notebook 2 of 5*

In notebook 1 you built a bag of trees and watched averaging melt their variance away — *roughly* like
σ²/B. That word "roughly" was a promise to keep, and there was a loose end: on `make_moons`,
scikit-learn's **default** forest scored a little *below* your hand-built bag. Both threads lead to the
same missing idea, the one that turns a bag of trees into a *forest*: **decorrelation**.

**Prerequisites.**
- **Notebook 1** of this chapter: bagging (bootstrap + majority vote) and the σ²/B intuition with its
  "roughly".
- **Chapter 04**: a single deep tree, and that it is scale-invariant (so no standardization here).
- **Module 00** (NB 10): cross-validation, which we use as the stable yardstick.

**What you'll be able to do.**
- Measure the **pairwise correlation ρ** between the trees in a forest.
- Derive the variance law **Var = ρσ² + (1−ρ)σ²/B** from scratch, and read its **floor**.
- See **feature subsampling** lower ρ and lift the ensemble — while the individual trees stay no better.
- Use **`max_features`** as the decorrelation dial, and explain why the "random" idea needs many
  features.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED = 0

# 30 features, many of them correlated (radius/perimeter/area, ...) — where feature
# subsampling can bite. Malignant = 1 (the costly miss), as in chapters 03-05.
data = load_breast_cancer()
X, y = data.data, (data.target == 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"breast cancer: train {X_train.shape}, test {X_test.shape}, {X.shape[1]} features")
print(f"malignant (train/test): {y_train.sum()}/{y_test.sum()};  sqrt(30) = {np.sqrt(30):.2f}")

## Two loose ends from notebook 1

Bagging averaged many bootstrap trees and the variance fell — but only *roughly* like σ²/B, and we
flagged why: that clean formula assumes the trees are **independent**, and bagged trees are not. They
are grown from overlapping resamples of the same data, so they tend to make the **same** mistakes.

The second loose end: on `make_moons`, the default `RandomForestClassifier` scored *below* our hand-bag
(0.900 vs 0.933). We parked that as "the one extra idea has little room on two features." Both loose
ends are about the same quantity — how alike the trees are — and the same fix. Time to name it and
measure it.

## Why bagging hits a ceiling

Picture the bagged trees as a committee where everyone read almost the same briefing. They will agree —
and when they are wrong, they tend to be wrong *together*. A vote only cleans up mistakes that
**differ** between members; mistakes they share sail straight through.

How alike are our bagged trees? We can measure it directly: take each tree's vector of predictions on
the test set and compute the average **pairwise correlation**, which we call **ρ** (rho). ρ near 0 means
the trees disagree freely (independent); ρ near 1 means they are near-copies. Let's read it off the bag
we built.

In [ ]:
def pairwise_corr(predictions):
    '''Mean Pearson correlation between every pair of trees' 0/1 prediction vectors.'''
    corr = []
    for i in range(len(predictions)):
        for j in range(i + 1, len(predictions)):
            a, b = predictions[i], predictions[j]
            if a.std() > 0 and b.std() > 0:
                corr.append(np.corrcoef(a, b)[0, 1])
    return float(np.mean(corr))


def fit_and_diversity(max_features, n_estimators=200, seed=SEED):
    '''Fit a forest; return (rho, mean individual-tree test accuracy).'''
    rf = RandomForestClassifier(
        n_estimators=n_estimators, max_features=max_features, random_state=seed, bootstrap=True
    ).fit(X_train, y_train)
    preds = np.array([tree.predict(X_test) for tree in rf.estimators_])
    indiv = np.mean([accuracy_score(y_test, p) for p in preds])
    return pairwise_corr(preds), indiv


def forest_cv(max_features, n_estimators=200, seed=SEED):
    '''Cross-validated accuracy of the whole forest on the training set.'''
    rf = RandomForestClassifier(
        n_estimators=n_estimators, max_features=max_features, random_state=seed
    )
    return cross_val_score(rf, X_train, y_train, cv=cv).mean()


# Bagging = a forest with feature subsampling OFF: every split may use all 30 features.
bag_rho, bag_indiv = fit_and_diversity(max_features=None)
bag_cv = forest_cv(max_features=None)
print("bagging (all 30 features per split):")
print(f"   pairwise tree correlation  rho = {bag_rho:.3f}")
print(f"   mean individual-tree accuracy = {bag_indiv:.3f}")
print(f"   ensemble CV-on-train accuracy = {bag_cv:.3f}")

**Read the result.** ρ ≈ **0.82** — the bagged trees are strongly correlated. Each one keeps splitting
first on the same few dominant measurements (tumour size and concavity), so they carve very similar
boundaries and err on the same patients. Their *individual* accuracy is fine (~0.91), but averaging such
look-alikes recovers far less than the σ²/B of an independent committee would. To see exactly how much ρ
costs us, we need the variance law with the correlation put back in.

(A note on measurement: we read ρ off the trees' 0/1 predictions — a practical proxy for the
error-correlation the law assumes, not identical to it, but tracking the same thing: how often the
trees are wrong together.)

## The variance law, with correlation put back in

Notebook 1's σ²/B assumed independence. Let's redo it honestly. Take B trees, each a predictor of
variance σ², with average pairwise **correlation ρ**. Expand the variance of their average: the sum has
**B variance terms** (each σ²) and **B(B−1) covariance terms** (each ρσ²), all divided by B²:

$$\operatorname{Var}\!\left(\tfrac{1}{B}\textstyle\sum_i X_i\right)
= \frac{1}{B^2}\big[\,B\,\sigma^2 + B(B-1)\,\rho\sigma^2\,\big]
= \rho\,\sigma^2 + \frac{1-\rho}{B}\,\sigma^2.$$

Read the two terms. The second, (1−ρ)σ²/B, **melts away** as B grows — that is the part bagging (more
trees) drives down. The first, **ρσ², does not depend on B at all**: it is a **floor** no number of
trees can break. With ρ = 0 we recover σ²/B (the independent ideal); with ρ = 0.82 the variance is stuck
near 0.82·σ² however many trees we add. The only way under the floor is to **lower ρ**.

In [ ]:
def variance_of_mean(rho, n_trees, trials=20000, seed=SEED):
    '''Empirical variance of the mean of n_trees unit-variance variables with pairwise corr rho.'''
    r = np.random.default_rng(seed)
    common = r.standard_normal((trials, 1))
    private = r.standard_normal((trials, n_trees))
    samples = np.sqrt(rho) * common + np.sqrt(1 - rho) * private  # var 1, pairwise corr rho
    return samples.mean(axis=1).var()


print(f"{'rho':>5} {'B':>5} {'empirical':>11} {'formula':>10}")
for rho in [0.0, 0.5, 0.8]:
    for n_trees in [5, 25, 100]:
        emp = variance_of_mean(rho, n_trees)
        print(f"{rho:5.1f} {n_trees:5d} {emp:11.4f} {rho + (1 - rho) / n_trees:10.4f}")

**Read it.** The Monte-Carlo numbers match the formula term for term. The story is in the floor: at
ρ = 0 the variance falls to σ²/B and keeps falling (0.01 at B = 100) — adding trees is enough. But at
ρ = 0.8 it settles near 0.8 and **stays there**, no matter how many trees we average. Our bag (ρ ≈ 0.82)
is parked barely above its own floor; piling on more trees will not move it. To do better we must make
the trees **disagree**.

## The "random" idea: subsample the features

Here is the one move that turns a bag into a forest. At **every split**, instead of letting the tree
pick the best of all 30 features, show it only a **random handful** (say √30 ≈ 5) and let it pick the
best of those. Different splits see different candidates, so the trees are pushed down different paths —
they grow different shapes and make different mistakes. Different mistakes mean **lower ρ**, and a lower
ρ means a lower variance floor.

The cost: each tree is a little hobbled (it cannot always reach for the single best feature), so on its
own it may be slightly weaker. The bet of the random forest is that **decorrelation buys more than
individual strength gives up**. Let's test that bet.

In [ ]:
# Same forest, but each split now sees only sqrt(30) ~ 5 randomly chosen features.
rf_rho, rf_indiv = fit_and_diversity(max_features="sqrt")
rf_cv = forest_cv(max_features="sqrt")

print(f"{'':22}{'bagging(all 30)':>16}{'RF(sqrt~5)':>14}")
print(f"{'pairwise corr rho':22}{bag_rho:>16.3f}{rf_rho:>14.3f}")
print(f"{'individual tree acc':22}{bag_indiv:>16.3f}{rf_indiv:>14.3f}")
print(f"{'ensemble CV accuracy':22}{bag_cv:>16.3f}{rf_cv:>14.3f}")

**Read the result — the heart of the random forest.** Two things here are robust, one is not.
**Robust:** feature subsampling pulled ρ from **0.82 down to 0.80** (it does so on every seed), and the
**individual trees are no better** — 0.91 either way. **Not robust:** the ensemble's CV *here* nudges
from 0.947 to 0.957, but on 171 patients that single gap sits inside a ±0.01 seed band — resample and
it can shrink or even flip.

So read the win by **elimination**, not by the size of the numbers. The trees did not get smarter (0.91
unchanged), so whatever averaging buys can only come from their **disagreeing more** — a lower ρ, hence
a lower variance *floor* (cell 7). On this near-ceiling problem the accuracy headroom is tiny, so the
floor barely binds and the accuracy change is small and noisy; the **reliable** prize is the
decorrelation itself, bought at no cost to the trees. (Bagging and `'sqrt'` ≈ 5 both sit on the flat
part of the individual-strength curve; push `max_features` lower and the trees *do* weaken — the next
figures show it.)

In [ ]:
# Sweep how many features each split may see, from 1 up to all 30 (= bagging).
mf_values = [1, 2, 3, 5, 7, 10, 15, 20, 30]
rho_curve, indiv_curve, cv_curve = [], [], []
for mf in mf_values:
    rho_mf, indiv_mf = fit_and_diversity(max_features=mf, n_estimators=120)
    rho_curve.append(rho_mf)
    indiv_curve.append(indiv_mf)
    cv_curve.append(forest_cv(max_features=mf, n_estimators=120))

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(mf_values, rho_curve, marker="o", color=COLORS["highlight"])
ax.axvline(5, color=COLORS["muted"], linestyle="--", linewidth=1)
ax.text(5.4, min(rho_curve), "'sqrt' default", color=COLORS["muted"], fontsize=10)
ax.text(30, rho_curve[-1] + 0.004, "all 30 = bagging", color=COLORS["text"],
        fontsize=10, ha="right")
ax.set_xlabel("max_features (candidates per split)")
ax.set_ylabel("mean pairwise tree correlation  ρ")
ax.set_title("max_features is the decorrelation dial")
fig.tight_layout()
print("rho by max_features:", [round(r, 3) for r in rho_curve])

**Read the figure.** ρ climbs steeply at first, then **saturates** near 0.82: once a split already sees
most of the 30 features, the trees can hardly grow more alike, so bagging (all 30) sits on that high-ρ
plateau at the right. Fewer candidates per split forces the trees onto different features, pushing ρ —
and the variance floor — down. `max_features` is the **decorrelation dial**: turn it down to push ρ
down.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(mf_values, cv_curve, marker="o", color=COLORS["model"], label="ensemble (CV accuracy)")
ax.plot(mf_values, indiv_curve, marker="s", color=COLORS["class_b"], label="mean individual tree")
ax.set_xlabel("max_features (candidates per split)")
ax.set_ylabel("accuracy")
ax.set_title("the committee beats its members — most when they disagree")
ax.legend()
fig.tight_layout()
print("ensemble CV: ", [round(float(c), 3) for c in cv_curve])
print("indiv  tree: ", [round(float(c), 3) for c in indiv_curve])

**Read the figure.** The two curves tell the whole story. The individual trees (lower curve) are
*weakest* when `max_features` is smallest — a tree shown one feature at a time is not much of a tree. Yet
the ensemble (upper curve) is *strongest* exactly there, because those weak trees are the most
decorrelated. **Decorrelation can outweigh individual strength.**

A caveat to carry: push `max_features` too low and the trees can get too weak to rescue, even
decorrelated; the curve here happens to peak at 1, but on other data it peaks higher. The safe default is
**`'sqrt'`**, and the honest move is to choose `max_features` by cross-validation (notebook 4).

## The moons puzzle, solved

Now notebook 1's loose end resolves itself. `make_moons` has only **two** features, so `'sqrt'` rounds to
**one feature per split** — each tree is forced onto a single coordinate at a time, badly hobbled, with
no spare features for decorrelation to make up the difference. There, feature subsampling **reliably
hurts**: `RF(sqrt)` 0.900 < bagging 0.933.

Breast cancer has **thirty** features, so `'sqrt'` ≈ 5 leaves each tree plenty to work with while still
forcing variety — there, subsampling decorrelates **at no cost** to the trees, and the ensemble is at
worst even with bagging (here 0.957 vs 0.947, within the seed band). The lesson: **feature subsampling
needs many features** — with few it backfires; with many it is free decorrelation, and its accuracy
payoff shows most on harder problems (the demanding case, notebook 5).

In [ ]:
# Few features (moons) vs many (breast cancer): does feature subsampling help?
Xm, ym = make_moons(n_samples=300, noise=0.30, random_state=SEED)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(
    Xm, ym, test_size=0.30, random_state=SEED, stratify=ym
)
moons_bag = RandomForestClassifier(n_estimators=200, max_features=None, random_state=SEED)
moons_bag.fit(Xm_tr, ym_tr)
moons_rf = RandomForestClassifier(n_estimators=200, random_state=SEED).fit(Xm_tr, ym_tr)
moons_bag_acc = accuracy_score(ym_te, moons_bag.predict(Xm_te))
moons_rf_acc = accuracy_score(ym_te, moons_rf.predict(Xm_te))

print("moons (2 features):")
print(f"   bagging (all 2): test {moons_bag_acc:.3f}")
print(f"   RF (sqrt -> 1):  test {moons_rf_acc:.3f}   <- few features: subsampling hurts")
print("breast cancer (30 features):")
print(f"   bagging (all 30): CV {bag_cv:.3f}")
print(f"   RF (sqrt -> 5):   CV {rf_cv:.3f}   <- many features: decorrelates for free")

## You have built a random forest

Put the two ideas together and you have the whole method: **bagging** (notebook 1: bootstrap many trees,
vote) **plus feature subsampling** (this notebook: each split sees a random subset, to decorrelate). That
is a random forest. `'sqrt'` is the robust default for `max_features`; we tune it, and the rest of the
knobs, in notebook 4.

One thing the bootstrap quietly handed us is still unused: every tree skipped ~37% of the data. Notebook
3 turns those skipped points into a free, honest accuracy estimate — **out-of-bag** error — with no test
set set aside.

## Your turn

1. **Agreement by hand (easy).** Two trees predict `[1, 0, 1, 1, 0]` and `[1, 0, 0, 1, 0]` on five
   points. On how many do they agree? A high agreement hints at a high ρ — why does that make their
   votes less useful *together*?

2. **Read the floor (medium).** Using `Var = ρσ² + (1−ρ)σ²/B` with σ² = 1 and ρ = 0.8, compute the
   variance at B = 10 and at B = 1000, and the floor as B → ∞. What *fraction of the remaining gap* to
   that floor does going from B = 10 to B = 1000 close — and what does that say about piling on more
   trees?

3. **Pick a `max_features` (harder).** Sweep `max_features` over `[1, 2, 5, 10, 20, 30]`, and for each
   plot both ρ and the ensemble's CV accuracy. Which value would you ship — and explain, from the two
   curves, why the literal best on this split (1) is not the safe default.

## What you built

- You measured the **pairwise correlation ρ** between a forest's trees, and saw bagged trees are
  strongly correlated (ρ ≈ 0.82).
- You **derived** the variance law `Var = ρσ² + (1−ρ)σ²/B` from scratch and confirmed it by simulation —
  and read its **floor**, ρσ², the wall no number of trees can pass.
- You saw **feature subsampling** lower ρ (0.82 → 0.80) **at no cost to individual-tree strength** (0.91
  either way) — the robust win; the ensemble's accuracy held (the small CV change on this split sits
  within the seed band), and a lower ρ is what lowers the variance floor.
- You used **`max_features`** as the decorrelation dial, and resolved notebook 1's moons puzzle: the
  "random" idea needs many features to pay off.

**Vocabulary you now own:** pairwise correlation ρ · decorrelation · feature subsampling · `max_features`
· the variance floor ρσ² · random forest.

## Going further (optional)

If decorrelation is so valuable, why stop at subsampling features? **Extremely Randomized Trees**
(`ExtraTreesClassifier`) go further: at each split they also pick the threshold *at random* rather than
optimizing it, pushing ρ lower still — sometimes a better ensemble, sometimes too much randomness. And
`max_features` carries its own bias–variance trade: smaller means more decorrelation but weaker, more
biased trees, so the best value is a balance you tune (notebook 4).

## References

- Ho, T. K. (1998). *The random subspace method for constructing decision forests.* IEEE TPAMI 20(8),
  832–844. DOI: [10.1109/34.709601](https://doi.org/10.1109/34.709601)
- Breiman, L. (2001). *Random Forests.* Machine Learning 45, 5–32.
  DOI: [10.1023/A:1010933404324](https://doi.org/10.1023/A:1010933404324)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §15.2 — the Var = ρσ² + (1−ρ)σ²/B law. DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning*
  (2nd ed.), §8.2. DOI: [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1)

*Previous: 01 — The wisdom of trees: averaging cuts variance. Next: 03 — Out-of-bag estimation.*